# Spark Tune - Databricks ML Pipeline Demo

End-to-end ML pipeline reading data from a Databricks catalog and demonstrating:

4. **Auto Feature Generator** - Auto Mathematical Feature Generator


In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

---
## 1. Load Data from Databricks Catalog

In [0]:
CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"
# TABLE_NAME = "credit_card_transactions"
TABLE_NAME = "hdfc_demo_bank_customers"

FEATURE_SCHEMA_NAME = "feature_store"
FEATURE_TABLE_NAME = f"{TABLE_NAME}_auto"

# Table Names
auto_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{FEATURE_TABLE_NAME}"

credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

# Reading Original Data
df = spark.read.table(credit_card_transactions_table_name)

print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
df.printSchema()

In [0]:
display(df.limit(5))

---
## 2. Problem Definition & Schema Validation

In [0]:
from backend.core.discovery import Problem, SchemaChecks

problem = Problem(
    # target="is_fraud",
    target="responded",
    type="classification",
    desired_result=1,
    # date_column="trans_date_trans_time"
    date_column="contact_timestamp"
)

schema_checker = SchemaChecks(dataframe=df, problem=problem)
schema_info = schema_checker.check()

print(f"Problem Type: {problem.type}")
print(f"Target Column: {problem.target}")
print(f"Desired Result: {problem.desired_result}")
print(f"\nSchema Summary:")
print(f"  Categorical columns: {len(schema_info['categorical'])}")
print(f"  Numerical columns: {len(schema_info['numerical'])}")
print(f"  Boolean columns: {len(schema_info['boolean'])}")

---
## 7. Auto Feature Generation

At this point `df` contains the original columns.

Now we generate additional features (interactions, binning, datetime
extraction) using AutoFeatureGenerator.

In [0]:
from backend.core.features.auto_feature_generator import AutoFeatureGenerator

# Re-create schema checker on the enriched DataFrame (includes featuretools + tsfresh columns)
schema_checker = SchemaChecks(dataframe=df, problem=problem)
schema_checker.check()

# List all columns to drop from the main DataFrame
# _cols_meta = ["trans_num", "trans_date_trans_time"]
_cols_meta = ["customer_id", "contact_timestamp"]
col_to_drop = df.columns
# print("Columns to Drop: ", col_to_drop, type(col_to_drop))
for _col in _cols_meta:
    # print(_col)
    col_to_drop.remove(_col)


feature_gen = AutoFeatureGenerator(
    schema_checks=schema_checker,
    problem=problem
)

# skip_numerical_col = ["unix_time", "merch_zipcode", "zip", "lat", "long", "cc_num", "merch_lat", "merch_long"]
skip_numerical_col = []

numerical_cols = schema_checker.get_typed_col(col_type="numerical")
for col in skip_numerical_col:
    if col in numerical_cols: numerical_cols.remove(col)
categorical_cols = schema_checker.get_typed_col(col_type="categorical")
datetime_cols = schema_checker.get_typed_col(col_type="datetime")

# Remove target from feature lists
for col_list in [numerical_cols, categorical_cols, datetime_cols]:
    if problem.target in col_list:
        col_list.remove(problem.target)

print(f"Enriched DataFrame before auto-generation: {len(df.columns)} columns")
print(f"  Numerical: {len(numerical_cols)}, Categorical: {len(categorical_cols)}, Datetime: {len(datetime_cols)}")

df_with_features = feature_gen.generate_all_features(
    include_numerical=True,
    include_interactions=True,
    include_binning=True,
    include_datetime=True,
    include_string=False,
    numerical_columns=numerical_cols,
    categorical_columns=categorical_cols,
    datetime_columns=datetime_cols
)

#Drop original columns
df_with_features = df_with_features.drop(*col_to_drop)

print(f"\nFEATURE GENERATION SUMMARY:")
print(f"  Input features (original): {len(df.columns)}")
print(f"  Total features after auto-generation: {len(df_with_features.columns)}")
print(f"  New features generated: {len(df_with_features.columns) - len(df.columns)}")


## Feature Table

Saving data in feature tables

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Create feature table with `trans_num` as the primary key.
# Take schema from DataFrame output by featuretool_features
customer_feature_table = fe.create_table(
  name=auto_feature_table_name,
  # primary_keys=['trans_num', 'trans_date_trans_time'],
  primary_keys=['customer_id', 'contact_timestamp'],
  timeseries_columns='contact_timestamp',
  schema=df_with_features.schema,
  description='Auto-generated features'
)


##Debugging

In [0]:
# from pyspark.sql.functions import col, count, when, expr
# spark.conf.set("spark.sql.ansi.enabled", "false")

# # Idehttps://adb-7405618144348512.12.azuredatabricks.net/editor/notebooks/519605393822541?contextId=folder%3A4067257149781110&o=7405618144348512$0ntify potential overflow columns
# df_with_features.printSchema()
# df_with_features.select([count(when(col(c).isNull(), 1)).alias(c) for c in df_with_features.columns]).show()

In [0]:
# spark.conf.set("spark.sql.ansi.enabled", "false")

fe.write_table(
  name=auto_feature_table_name,
  df = df_with_features,
  mode = 'merge'
)
